为什么不用requests
接口包含passwor   logid   sign   type
是典型的前端签名校验系统，除非逆向JS，否则无法模拟

In [4]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains

def select_time_range(text):
    print(f"选择时间范围：{text}")

    # 1. 点击下拉框输入区域
    select_input = wait.until(EC.element_to_be_clickable((
        By.XPATH, "//form[contains(@class,'select_time')]//input[@placeholder='请选择']"
    )))
    driver.execute_script("arguments[0].click();", select_input)

    # 2. 等待下拉面板出现（注意：面板在body下）
    option = wait.until(EC.element_to_be_clickable((
        By.XPATH, f"//div[contains(@class,'el-select-dropdown')]//span[text()='{text}']"
    )))

    driver.execute_script("arguments[0].click();", option)

    # 3. 等待页面数据刷新（关键）
    time.sleep(2)

# ================= 配置 =================
USERNAME = "xq5"
PASSWORD = "123456"
BASE_URL = "https://bll.lingdongsz.com/uranus/#/login"

chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=chrome_options)
driver.maximize_window()
wait = WebDriverWait(driver, 30)
actions = ActionChains(driver)

# ================= 登录 =================
print("打开登录页")
driver.get(BASE_URL)

wait.until(EC.presence_of_element_located((By.XPATH, '//input[@type="text"]'))).send_keys(USERNAME)
driver.find_element(By.XPATH, '//input[@type="password"]').send_keys(PASSWORD)
driver.find_element(By.XPATH, '//button').click()

time.sleep(2)
print("登录成功")

# ================= 进入美客多 =================
print("进入美客多")

mercado_btn = wait.until(
    EC.element_to_be_clickable((By.ID, "mercadoId"))
)
mercado_btn.click()

time.sleep(3)

# 清理弹窗
driver.execute_script("""
let dialogs = document.querySelectorAll('.el-dialog__wrapper');
dialogs.forEach(d => d.remove());
let masks = document.querySelectorAll('.v-modal');
masks.forEach(m => m.remove());
""")

# ================= 进入商品分析 =================
print("进入商品分析")

menu_btn = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//span[contains(text(),'商品分析')]")
))
menu_btn.click()

time.sleep(6)

# ================= 点击导出明细 =================
#print("点击导出明细")

export_btn = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//span[contains(text(),'导出明细')]")
))
export_btn.click()

print("商品分析导出任务已提交，等待5分钟生成文件...")

# ================= 进入历史分析 =================
#print("进入交易分析")

user_center = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//span[contains(text(),'交易分析')]")
))
user_center.click()

download_center = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//span[contains(text(),'历史分析')]")
))
download_center.click()

time.sleep(5)

print("进入历史分析，等待页面加载")

# ================= 历史分析：点击导出明细 =================
#print("点击导出明细(历史分析)")
# 选择近7天
select_time_range("近7天")
time.sleep(2)
# ================= 历史分析：点击 SKU每日表现 =================
print("点击 SKU每日表现")

try:
    sku_btn = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((
            By.XPATH, "//button[.//span[contains(text(),'SKU每日表现')]]"
        ))
    )

    # 滚动到可见位置
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", sku_btn)
    time.sleep(1)

    # 用JS点击（Element弹层页面推荐）
    driver.execute_script("arguments[0].click();", sku_btn)

    print("SKU每日表现")

    # 等待页面数据刷新
    time.sleep(3)

except Exception as e:
    print("未找到 SKU每日表现 按钮:", e)

export_btn = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//span[contains(text(),'导出明细')]")
))
export_btn.click()

# ================= 历史分析：处理弹窗 =================
#print("等待历史分析弹窗，点击加入报表下载")
time.sleep(2) # 等待弹窗渲染动画
try:
    # 匹配包含文本“加入报表下载”的按钮
    report_btn = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//button[contains(., '加入报表下载')]")
    ))
    report_btn.click()
    print("历史分析导出任务已提交，等待5分钟生成文件...")
    time.sleep(1) # 等待弹窗消失

    # 【核心修复】：强制干掉所有全局遮罩层，防止页面假死变灰
    driver.execute_script("""
        let masks = document.querySelectorAll('.v-modal');
        masks.forEach(m => m.remove());
        let dialogs = document.querySelectorAll('.el-dialog__wrapper');
        dialogs.forEach(d => { if(d.style.display !== 'none') d.style.display = 'none'; });
    """)
    print("已强制清理历史分析的遮罩层")

except Exception as e:
    print("未找到'加入报表下载'按钮，跳过或检查页面:", e)
# ==============================================================

print("进入广告流量，等待页面加载")
ads_menu = wait.until(EC.presence_of_element_located(
    (By.XPATH, "//span[contains(text(),'广告流量')]")
))
driver.execute_script("arguments[0].click();", ads_menu)

time.sleep(5)

# ================= 点击导出明细7天 =================
#print("点击导出明细")
select_time_range("近7天")

export_btn = wait.until(EC.presence_of_element_located((
    By.XPATH, "//button[.//span[contains(text(),'导出明细')]]"
)))

driver.execute_script("arguments[0].scrollIntoView({block:'center'});", export_btn)
time.sleep(1)

ActionChains(driver).move_to_element(export_btn).pause(0.5).click().perform()

#print("导出按钮已点击")
time.sleep(1)
# ================= 等待MessageBox弹出 =================
#print("等待确认弹窗(el-message-box)...")

try:
    msg_box = WebDriverWait(driver, 30).until(
        EC.visibility_of_element_located((
            By.CLASS_NAME, "el-message-box__wrapper"
        ))
    )

    confirm_btn = msg_box.find_element(
        By.XPATH, ".//button[contains(@class,'el-button--primary')]"
    )

    driver.execute_script("arguments[0].click();", confirm_btn)

    print("广告流量文件导出成功")

except Exception as e:
    print("未检测到确认弹窗:", e)

# 再次清理遮罩（防止点击被挡）
driver.execute_script("""
let masks = document.querySelectorAll('.v-modal');
masks.forEach(m => m.remove());
""")

# ================= 点击导出明细30天 =================
#print("点击导出明细")
select_time_range("近30天")

export_btn = wait.until(EC.presence_of_element_located((
    By.XPATH, "//button[.//span[contains(text(),'导出明细')]]"
)))

driver.execute_script("arguments[0].scrollIntoView({block:'center'});", export_btn)
time.sleep(1)

ActionChains(driver).move_to_element(export_btn).pause(0.5).click().perform()

#print("导出按钮已点击")
time.sleep(1)
# ================= 等待MessageBox弹出 =================
#print("等待确认弹窗(el-message-box)...")

try:
    msg_box = WebDriverWait(driver, 30).until(
        EC.visibility_of_element_located((
            By.CLASS_NAME, "el-message-box__wrapper"
        ))
    )

    confirm_btn = msg_box.find_element(
        By.XPATH, ".//button[contains(@class,'el-button--primary')]"
    )

    driver.execute_script("arguments[0].click();", confirm_btn)

    print("广告流量文件导出成功")

except Exception as e:
    print("未检测到确认弹窗:", e)

# ==============================================================

# 【核心修复】：强制干掉所有全局遮罩层，防止页面假死变灰
driver.execute_script("""
    let masks = document.querySelectorAll('.v-modal');
    masks.forEach(m => m.remove());
    let dialogs = document.querySelectorAll('.el-dialog__wrapper');
    dialogs.forEach(d => { if(d.style.display !== 'none') d.style.display = 'none'; });
""")
#print("已强制清理历史分析的遮罩层")

# ================= 等待5分钟 =================
print("准备等待5分钟让文件生成...")
time.sleep(300)   # 固定等待5分钟

# ================= 进入下载中心 =================
print("进入个人中心")

user_center = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//span[contains(text(),'个人中心')]")
))
user_center.click()

time.sleep(2)

download_center = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//span[contains(text(),'下载中心')]")
))
download_center.click()

print("进入下载中心，等待页面加载")

# 等待页面出现下载按钮
wait.until(EC.presence_of_element_located(
    (By.XPATH, "//span[text()='下载']")
))

time.sleep(2)

# 再次清理遮罩（防止点击被挡）
driver.execute_script("""
let masks = document.querySelectorAll('.v-modal');
masks.forEach(m => m.remove());
""")

# ================= 点击前两个下载按钮 =================
for i in range(1, 3):
    try:
        print(f"下载第{i}个文件")
        btn = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f"(//span[text()='下载'])[{i}]")
        ))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        time.sleep(1)
        actions.move_to_element(btn).pause(0.3).click().perform()
        time.sleep(2)
    except Exception as e:
        print(f"第{i}个下载失败:", e)

# 等待下载开始
time.sleep(1)

driver.quit()
print("流程结束")

打开登录页
登录成功
进入美客多
进入商品分析
商品分析导出任务已提交，等待5分钟生成文件...
进入历史分析，等待页面加载
选择时间范围：近7天
点击 SKU每日表现
SKU每日表现
历史分析导出任务已提交，等待5分钟生成文件...
已强制清理历史分析的遮罩层
进入广告流量，等待页面加载
选择时间范围：近7天
广告流量文件导出成功
选择时间范围：近30天
广告流量文件导出成功
准备等待5分钟让文件生成...
进入个人中心
进入下载中心，等待页面加载
下载第1个文件
下载第2个文件
流程结束


In [1]:
import os
import pandas as pd
from openpyxl import load_workbook

# ========= 配置 =========
DOWNLOAD_DIR = r"C:\Users\kangjia\Downloads"
TEMPLATE_PATH = r"C:\Users\kangjia\Desktop\新运营表_XQ5.xlsx"
SHEET_NAME = "数据更新表"


# ==========================================================
# 工具函数
# ==========================================================

# 获取最新文件（按关键词）
def get_latest_file(folder, keyword):
    files = []
    for f in os.listdir(folder):
        if f.endswith((".xlsx", ".xls")) and keyword in f:
            files.append(os.path.join(folder, f))

    if not files:
        raise Exception(f"未找到包含【{keyword}】的文件")

    latest_file = max(files, key=os.path.getmtime)
    print(f"使用{keyword}文件：", latest_file)
    return latest_file


# 获取两个最新广告文件（30天 / 7天）
def get_latest_ads_files(folder):
    files = []
    for f in os.listdir(folder):
        if f.endswith((".xlsx", ".xls")) and "广告流量" in f:
            files.append(os.path.join(folder, f))

    if len(files) < 2:
        raise Exception("广告流量文件少于2个")

    files_sorted = sorted(files, key=os.path.getmtime, reverse=True)

    file_30 = files_sorted[0]   # 最新 → 30天
    file_7 = files_sorted[1]    # 次新 → 7天

    print("30天广告文件：", file_30)
    print("7天广告文件：", file_7)

    return file_30, file_7


# 通用写入函数（提高可读性）
def write_dataframe(ws, df, columns, start_col, start_row=2):
    data = df[columns]
    print(f"写入列 {start_col}，行数：", len(data))

    for i, row in data.iterrows():
        r = start_row + i
        for j, col in enumerate(columns):
            ws.cell(r, start_col + j).value = row[col]


# ==========================================================
# 主流程
# ==========================================================

print("\n========== 开始更新运营表 ==========")

# 打开模板（只打开一次）
wb = load_workbook(TEMPLATE_PATH)
ws = wb[SHEET_NAME]


# ==========================================================
# 一、历史分析 → E–G（5–7列）
# ==========================================================
history_file = get_latest_file(DOWNLOAD_DIR, "历史分析")
df_history = pd.read_excel(history_file)

write_dataframe(
    ws,
    df_history,
    ["SKU", "近7日销量", "昨日销量"],
    start_col=5
)

print("历史分析已更新（E–G）")


# ==========================================================
# 二、商品分析 → J–N（10–14列）
# ==========================================================
product_file = get_latest_file(DOWNLOAD_DIR, "商品分析")
df_product = pd.read_excel(product_file)

write_dataframe(
    ws,
    df_product,
    [
        "商品ID",
        "购物体验分",
        "折扣价(MXN)",
        "原价(MXN)",
        "近7天访问量"
    ],
    start_col=10
)

print("商品分析已更新（J–N）")


# ==========================================================
# 三、广告流量
# ==========================================================
file_30, file_7 = get_latest_ads_files(DOWNLOAD_DIR)

# -------- 7天广告 → Z–AJ（26–36列） --------
df_7 = pd.read_excel(file_7)

write_dataframe(
    ws,
    df_7,
    [
        "商品ID",
        "展现量",
        "点击数",
        "ACoS",
        "广告花费(MXN)",
        "广告销量",
        "直接销量",
        "间接销量",
        "自然销量",
        "TACOS(ACOAS)",
        "ROAS"
    ],
    start_col=26
)

print("7天广告已更新（Z–AJ）")


# -------- 30天广告 → AL–AM（38–39列） --------
df_30 = pd.read_excel(file_30)

write_dataframe(
    ws,
    df_30,
    [
        "商品ID",
        "TACOS(ACOAS)"
    ],
    start_col=38
)

print("30天广告已更新（AL–AM）")


# ==========================================================
# 保存
# ==========================================================
wb.save(TEMPLATE_PATH)

print("\n========== 全部更新完成 ==========")
print("文件位置：", TEMPLATE_PATH)


========== 开始更新运营表 ==========
使用历史分析文件： C:\Users\kangjia\Downloads\历史分析-SKU每日表现-20260302092838.xlsx
写入列 5，行数： 21
历史分析已更新（E–G）
使用商品分析文件： C:\Users\kangjia\Downloads\商品分析-商品明细-20260302092821.xlsx
写入列 10，行数： 31
商品分析已更新（J–N）
30天广告文件： C:\Users\kangjia\Downloads\广告流量-商品销售明细-20260302092704.xlsx
7天广告文件： C:\Users\kangjia\Downloads\广告流量-商品销售明细-20260302092659.xlsx
写入列 26，行数： 31
7天广告已更新（Z–AJ）
写入列 38，行数： 35
30天广告已更新（AL–AM）


D:\Program Files\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
D:\Program Files\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
D:\Program Files\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
D:\Program Files\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



========== 全部更新完成 ==========
文件位置： C:\Users\kangjia\Desktop\新运营表_XQ5.xlsx


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pyautogui
import pandas as pd
import pyperclip
import xlwings as xw

# ======================
# 本地文件配置
# ======================
LOCAL_FILE = r"C:\Users\kangjia\Desktop\新运营表_XQ5.xlsx"
LOCAL_SHEET = "数据更新表"

def copy_local_excel_to_clipboard():
    print("正在读取本地Excel...")

    # 1. 依然使用pandas读取纯数据（如果你的程序后续还需要用到 df 对象）
    df = pd.read_excel(LOCAL_FILE, sheet_name=LOCAL_SHEET)
    print(f"读取完成，行数: {len(df)}，列数: {len(df.columns)}")

    # 2. 启动 xlwings 调用真实的表格应用来复制，以保留所有格式
    print("正在调用底层应用复制数据与格式...")
    # visible=False 表示在后台静默打开，不弹出应用窗口
    app = xw.App(visible=False, add_book=False)

    try:
        wb = app.books.open(LOCAL_FILE)
        sht = wb.sheets[LOCAL_SHEET]

        # 选中该 sheet 中所有使用过的区域，并执行“复制”操作
        sht.used_range.copy()

        print("数据及格式已成功复制到系统剪贴板！")
        time.sleep(1)

    except Exception as e:
        print(f"复制过程中发生错误: {e}")

    return df, app, wb  # 把 app 和 wb 返回，方便你在后续粘贴完成后关闭它们

# ======================
# 参数配置
# ======================
URL = "https://www.kdocs.cn/"
SEARCH_NAME = "新运营表"
TARGET_FILE = "新运营表_XQ5.xlsx"

# ======================
# 启动浏览器
# ======================
def start_browser():
    options = Options()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 30)
    return driver, wait


# ======================
# 点击【免费使用】
# ======================
def click_free_use(driver, wait):
    driver.get(URL)

    free_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[text()='免费使用']")
        )
    )
    free_btn.click()
    print("已点击 免费使用")


# ======================
# 点击【确认登录】
# ======================
def confirm_login(driver, wait):
    confirm_btn = wait.until(
        EC.element_to_be_clickable(
            (By.ID, "confirmSync")
        )
    )
    confirm_btn.click()
    print("已确认登录")

    # 等待进入云文档首页
    time.sleep(5)

# ======================
# 鼠标点击关闭弹窗
# ======================
def close_popup_by_mouse():
    print("鼠标关闭弹窗...")
    pyautogui.moveTo(1095,451, duration=0.5)
    pyautogui.click()
    time.sleep(2)

# ======================
# 搜索文档
# ======================
def search_file(driver, wait):
    search_box = wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "input.kdv-input__inner")
        )
    )
    search_box.clear()
    search_box.send_keys(SEARCH_NAME)
    search_box.send_keys(Keys.ENTER)

    print("已搜索:", SEARCH_NAME)
    time.sleep(5)


# ======================
# 双击打开文件
# ======================
def open_excel(driver, wait):
    file_item = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, f"//*[contains(text(), '{TARGET_FILE}')]")
        )
    )

    ActionChains(driver).double_click(file_item).perform()
    print("已打开文件:", TARGET_FILE)

    # 等待表格加载
    time.sleep(10)

# ======================
# 鼠标双击打开文件
# ======================
def open_file_by_mouse():
    print("鼠标双击打开文件...")
    pyautogui.moveTo(678,412, duration=0.5)
    pyautogui.doubleClick()
    time.sleep(8)

# 为防止误操作，执行前有1秒缓冲
pyautogui.FAILSAFE = True  # 鼠标移动到左上角可强制终止

def update_wps_by_coordinates():
    print("等待文件加载...")
    time.sleep(10)

    # Step 1：点击 (195,1017)
    print("点击位置1")
    pyautogui.moveTo(195, 1017, duration=0.4)
    pyautogui.click()
    time.sleep(1)

    # Step 2：点击 (137,405)
    print("点击粘贴起始单元格")
    pyautogui.moveTo(137, 405, duration=0.4)
    pyautogui.click()
    time.sleep(1)

    # Step 3：Ctrl + V 粘贴
    print("粘贴数据")
    pyautogui.hotkey('ctrl', 'v')
    time.sleep(3)

    print("数据更新完成")

# ======================
# 主流程
# ======================
if __name__ == "__main__":
    driver, wait = start_browser()

    click_free_use(driver, wait)
    confirm_login(driver, wait)

    close_popup_by_mouse()
    search_file(driver, wait)

    open_file_by_mouse()
    print("线上文件已打开")

    # 【新增逻辑 1】：在准备点击和粘贴之前，调用函数将本地Excel数据写入剪贴板
    print("========== 开始读取并复制本地数据 ==========")
    df, excel_app, excel_wb = copy_local_excel_to_clipboard()

    # 文件打开后更新数据 (执行鼠标点击、清空、Ctrl+V)
    print("========== 开始执行网页端粘贴更新 ==========")
    update_wps_by_coordinates()

    driver.quit()
    print("流程结束")

已点击 免费使用
已确认登录
鼠标关闭弹窗...
已搜索: 新运营表
鼠标双击打开文件...
线上文件已打开
========== 开始读取并复制本地数据 ==========
正在读取本地Excel...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\kangjia\\Desktop\\新运营表_XQ5.xlsx'

获取坐标

In [23]:
import pyautogui
import time

print("5秒后开始获取坐标")
time.sleep(5)

while True:
    x, y = pyautogui.position()
    print(x, y)
    time.sleep(1)

5秒后开始获取坐标
195 1017
195 1017
195 1017
195 1017
195 1017
195 1017
195 1017
195 1017
195 1017
195 1017
189 1017
51 335
18 372
25 371
18 370
24 372
23 372
21 371
21 371
21 371
21 371
21 371
21 371
21 371
21 371
21 371
136 405
137 405
137 405
137 405
137 405
137 405
137 405
137 405
137 405
352 630
1048 441


KeyboardInterrupt: 